# Interview Transcriber — clean-room launcher

Run these cells top to bottom in a fresh Google Colab runtime with a T4 GPU. `HF_TOKEN` must exist in Colab Secrets and be granted to this notebook. The token is never printed or written to Drive.

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/transcriber')
if (REPO_DIR / '.git').exists():
    !git -C /content/transcriber fetch --quiet origin main
    !git -C /content/transcriber reset --hard origin/main
else:
    !git clone --branch main https://github.com/playply/transcriber.git /content/transcriber

%cd /content/transcriber
!python -m pip install -q -r requirements.txt


If the next cell fails with a NumPy/import compatibility error immediately after installation, restart the Colab runtime once and rerun from the first cell. This was observed in a prior Colab runtime transition.

In [ ]:
import sys
import torch
import whisperx
import transformers

assert torch.cuda.is_available(), 'CUDA is unavailable. Select a Colab T4 GPU runtime.'
gpu_name = torch.cuda.get_device_name(0)
print('Python:', sys.version.split()[0])
print('GPU:', gpu_name)
print('Torch:', torch.__version__)
print('WhisperX:', getattr(whisperx, '__version__', 'unknown'))
print('Transformers:', transformers.__version__)


In [ ]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN is missing from Colab Secrets')
os.environ['HF_TOKEN'] = hf_token
del hf_token
print('Drive mounted; HF_TOKEN loaded from Colab Secrets.')


In [ ]:
from pathlib import Path
import subprocess
import sys

SOURCE_PATH = '/content/drive/MyDrive/Interviews/meeting.m4a'  # change only this line
source = Path(SOURCE_PATH)
assert source.is_file(), f'Source recording not found: {source}'
assert source.suffix.lower() in {'.mp4', '.mp3', '.m4a', '.wav'}, f'Unsupported format: {source.suffix}'

subprocess.run([sys.executable, 'app.py', str(source)], check=True)


In [ ]:
from pathlib import Path

source = Path(SOURCE_PATH)
base = source.with_suffix('')
expected = [
    Path(f'{base}_transcript.docx'),
    Path(f'{base}_transcript.txt'),
    Path(f'{base}_transcript.json'),
]
missing = [str(path) for path in expected if not path.is_file()]
assert not missing, f'Missing expected outputs: {missing}'
print('Clean-room baseline outputs verified:')
for path in expected:
    print(path)
